In [3]:
pip install datasets


   ---------------------------------------- 0.0/116.3 kB ? eta -:--:--
   --- ------------------------------------ 10.2/116.3 kB ? eta -:--:--
   ---------- ---------------------------- 30.7/116.3 kB 640.0 kB/s eta 0:00:01
   ------------- ------------------------- 41.0/116.3 kB 393.8 kB/s eta 0:00:01
   ------------------------ -------------- 71.7/116.3 kB 435.7 kB/s eta 0:00:01
   --------------------------------- ---- 102.4/116.3 kB 490.2 kB/s eta 0:00:01
   -------------------------------------- 116.3/116.3 kB 484.1 kB/s eta 0:00:00


### Load Dataset 

In [53]:
from datasets import load_dataset

dataset = load_dataset("Divyaamith/Kaggle-Resume")


Repo card metadata block was not found. Setting CardData to empty.


In [55]:
import pandas as pd

df = dataset['train'].to_pandas()

print(df.shape)
df.head()


(2484, 4)


,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


### Text Cleaning

In [14]:
pip install nltk


     ---------------------------------------- 0.0/41.5 kB ? eta -:--:--
     --------- ------------------------------ 10.2/41.5 kB ? eta -:--:--
     ------------------ ------------------- 20.5/41.5 kB 320.0 kB/s eta 0:00:01
     ------------------ ------------------- 20.5/41.5 kB 320.0 kB/s eta 0:00:01
     ------------------ ------------------- 20.5/41.5 kB 320.0 kB/s eta 0:00:01
     ------------------ ------------------- 20.5/41.5 kB 320.0 kB/s eta 0:00:01
     ------------------ ------------------- 20.5/41.5 kB 320.0 kB/s eta 0:00:01
     ------------------ ------------------- 20.5/41.5 kB 320.0 kB/s eta 0:00:01
     ------------------ ------------------- 20.5/41.5 kB 320.0 kB/s eta 0:00:01
     ------------------ ------------------- 20.5/41.5 kB 320.0 kB/s eta 0:00:01
     ------------------ ------------------- 20.5/41.5 kB 320.0 kB/s eta 0:00:01
     ------------------ ------------------- 20.5/41.5 kB 320.0 kB/s eta 0:00:01
     ------------------ ------------------- 20.5/41.5 

In [57]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\SUSHANT\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\SUSHANT\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [59]:
def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return " ".join(words)


In [61]:
df['clean_resume'] = df['Resume_str'].apply(preprocess)

### TF-IDF

In [63]:
from sklearn.feature_extraction.text import TfidfVectorizer

job_description = """
Looking for Data Scientist with Python, Machine Learning,
SQL, NLP, Deep Learning and Statistics.
"""

clean_job = preprocess(job_description)

vectorizer = TfidfVectorizer(max_features=5000)
tfidf_matrix = vectorizer.fit_transform(df['clean_resume'])
job_vector = vectorizer.transform([clean_job])

### Cosine similarity

In [65]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(job_vector, tfidf_matrix)[0]
df['similarity_score'] = similarity

ranked = df.sort_values(by='similarity_score', ascending=False)
ranked[['Category','similarity_score']].head(10)

,Category,similarity_score
1762,ENGINEERING,0.237095
2153,BANKING,0.203284
2291,ARTS,0.183669
1142,CONSULTANT,0.178427
1218,CONSULTANT,0.177249
194,DESIGNER,0.166528
1348,AUTOMOBILE,0.157394
1339,AUTOMOBILE,0.144956
929,AGRICULTURE,0.142590
1303,DIGITAL-MEDIA,0.129487


### Skill Extraction

In [67]:
skill_list = ["python","java","sql","machine learning","deep learning","nlp","data analysis","excel","tableau","power bi","tensorflow","pytorch","c++"]


#skill matching 

In [69]:
def extract_skills(text):
    found_skills = []
    for skill in skill_list:
        if skill in text:
            found_skills.append(skill)
    return found_skills

df['skills'] = df['clean_resume'].apply(extract_skills)


In [73]:
df['skill_count'] = df['skills'].apply(len)

In [75]:
df['skill_score'] = df['skill_count'] / df['skill_count'].max()


### Weighted scoring system 

In [77]:
df['final_score'] = (0.8 * df['similarity_score']) + (0.2 * df['skill_score'])

ranked_final = df.sort_values(by='final_score', ascending=False)
ranked_final[['Category','final_score']].head(10)


,Category,final_score
1762,ENGINEERING,0.389676
1348,AUTOMOBILE,0.325915
1218,CONSULTANT,0.308466
1303,DIGITAL-MEDIA,0.303590
926,AGRICULTURE,0.303216
1339,AUTOMOBILE,0.282631
1142,CONSULTANT,0.276075
2153,BANKING,0.262627
331,INFORMATION-TECHNOLOGY,0.258226
662,BUSINESS-DEVELOPMENT,0.250533
